# Data Communication and Poster Design for Evidence-Driven Research

**Companion to Topic 10 · Studio 10: Prepare to publish or present**

<hr>

<center>
<div>
<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" width="300"/>
</div>
</center>


# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb10b_data_communication_poster_student.ipynb)

---

## 🧭 How This Companion Fits Week 10

This week's main notebook, **nb10**, carries your bounded claim into a venue: the
venue contract, the five audits, compression for speech, and the defend-or-concede
call. This companion works on the craft underneath all of it. The first half
teaches six principles of data communication on real course data. The second half
walks the architecture of an evidence-driven research poster, block by block, for
the Purdue Fall Undergraduate Research Expo.

It reads alongside EDR|AI **ch. 32, Research Posters**, and **ch. 33, Poster
Criticism**, and it never replaces them. Nothing in it is collected or graded. Its
two exercises feed work you are already doing: the figure audit feeds the headline
figure of your M10 artifact, and the poster outline and abstract feed the poster
draft you build on the course template for M10, its revision at M11, and the
terminal lock at M13.

| | |
|---|---|
| **A claim this companion PERMITS** | "My figure and my poster say exactly what my bounded note says: same verb, same reach, the uncertainty visible in the same glance, and every reader able to receive it." |
| **A claim this companion does NOT permit** | A design choice that makes the finding look bigger, more certain, more causal, or more general than the evidence behind it. Good design makes a true claim easy to see. It never makes a claim truer. |

## Learning Objectives

By the end of this notebook, you will be able to:

1. Apply the **six principles of data communication** (context, visualization,
   less is more, hierarchy, beauty, story) to a figure from your own project.
2. Diagnose common chart failures (a truncated scale, dual axes, a pie for ordered
   data, a table where a graph belongs) and rebuild the same numbers into an
   honest view.
3. Raise a figure's **data-ink ratio** without deleting the comparison, the
   uncertainty, or the boundary, which always count as data.
4. Plan the **layout, typography, and read path** of a research poster that a
   stranger reads in ninety seconds from four feet away.
5. Outline your poster's blocks and update your structured abstract so the
   headline claim, its uncertainty, and its boundary survive the move to the wall.

---

## 🤖 Your AI Research Partner in This Notebook

Your AI is useful here in three jobs you can check: generating the hardest
questions a skeptic would ask of a figure, scanning your plotting code for
distinctions carried by color alone, and listing every word in a headline or an
abstract that reaches further than your evidence. Each prompt below hands it one of
those jobs and ends with an **After running, verify** checklist and a ledger line.

The conference draws its own line, and it matches this course's. The Expo's AI
guidance accepts brainstorming, editing support, practice questions, and checks of
citation format. It rejects AI-generated sections of a poster or an abstract,
automated poster design without substantial human input, and any fabricated or
misrepresented analysis. It also asks you to disclose the tool, its purpose, and
how you changed its output. Its core test is the one EDR|AI uses: you must be able
to explain and defend every element on your board without the tool beside you.
Read it once in full:
[Fall Expo poster guidelines](https://www.purdue.edu/undergrad-research/conferences/fall/guidelines.php).

**Do not delegate.** The words of your headline, the verb that keeps it inside your
compass position, the judgment that a figure's impression matches its number, your
title and abstract, and the call that the poster is honest enough to lock. These
stay yours.

## Why This Matters

> *"I judge thirty posters in an afternoon. I read the title, then the biggest
> figure, then one sentence. If the figure looks like a landslide and the sentence
> says 'slightly higher,' I stop trusting the board, and I move on."*
> A **faculty judge** at an undergraduate research expo

You have spent nine studios building a claim you can defend: a question declared on
the inquiry compass, a design that licenses an answer, data you governed, an
analysis anyone can rerun, and a bounded note with a claim-evidence table behind
it. The work is real. The next weeks decide whether anyone outside this room ever
sees it, and whether what they see is the claim you earned or a louder one.

A visitor at the Expo gives you about ninety seconds before deciding to lean in or
walk on. They will not read your code or open your notebook. They will read your
**title**, your **headline figure**, and **one sentence**. If those three reveal a
clear question, a bounded answer, and an uncertainty they can repeat, you win the
next two minutes. If they reveal a busy chart and a verb your design never paid
for, you lose the visitor, and a judge who catches the overreach discounts every
other number on the board.

> **A question that often comes up here:** *"Is this about research, or about
> making things pretty?"* It is about research. A figure is the channel your
> evidence travels through, and a channel can distort. A floored axis, a rainbow
> palette, or a missing interval changes what a reader concludes without changing a
> single number. So treat design the way you treated measurement in Studio 6: as
> something you check against a standard, not a coat of paint at the end. The
> standard is simple to state and hard to meet. The picture may make the claim
> easier to see. It may never make the claim bigger.

---

### ⚙️ Setup

Run the cell below first. It loads the two course files this notebook draws from,
sets the seed, and defines one small helper that removes the borders every figure
here does without. Expect a single line confirming both files loaded.

In [ ]:
# Setup: run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11, "axes.titlesize": 12})

SEED = 464  # course number: keeps every simulation and subsample reproducible
rng = np.random.default_rng(SEED)

# The two colors this notebook spends: a quiet grey for context, and ONE accent.
GREY, DARK, ACCENT = "#B0B0B0", "#4A4A4A", "#0072B2"   # the accent blue is color-blind safe

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

def declutter(ax, keep_left=False):
    """Remove the non-data ink (frame and tick marks) every figure here does without."""
    for side in ["top", "right"] + ([] if keep_left else ["left"]):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=0)

lapop = load_course_data("lapop_brazil.csv")
foos = load_course_data("foos_etal.csv")
assert lapop.shape == (10000, 10) and foos.shape == (8375, 5), "unexpected shape: flag this!"
print("✓ Setup complete, seed", SEED, "| lapop_brazil", lapop.shape, "| foos_etal", foos.shape)


**Two files, two caveats that travel with every number.** `lapop_brazil` is a
10,000-row resample with replacement of real AmericasBarometer interviews from
Brazil: realistic in its patterns, manufactured in its precision. It serves
practice and never a claim about Brazil, so the cells below draw seeded,
project-sized subsamples from it. `foos_etal` is the replication file of a real
get-out-the-vote field experiment in the United Kingdom, in which whole streets were
randomly assigned to receive the treatment.

*Dataset from the `rdss` package (Blair, Coppock & Humphreys, MIT License),
companion to* Research Design in the Social Sciences *(2023).*

---

## 1. The Forest and the Trees

For nine studios you have worked close to the ground: a kind-and-reach
declaration, a pathway, measurement decisions, a first reproducible analysis, a
stress test, and a bounded note. This week the camera pulls back. The question is
no longer *"is my analysis right?"* but *"does my claim stay right, and stay
**legible**, when a stranger meets it on a wall?"*

That shift matters more than it sounds. An estimate of half a point on a 1-to-7
scale, with its interval and a clear boundary, is a real finding. Drawn on a floored
axis under a causal headline, the same half point becomes a claim nobody earned.
Getting the analysis right was the hard part. Keeping it right in public is this
week's work.

> **A question that often comes up here:** *"Why learn design now instead of a
> little each week?"* Because choices that look harmless inside a notebook (a chart
> with two y-axes, an eleven-column table, a caption that says "see code") become
> expensive when they are the **only** thing a reader sees. Deciding what stays,
> what goes, and what becomes the headline is far harder once the poster is
> half-built. So you decide it before you open the template.

---

## 2. The Six Principles of Data Communication

Each of the next six sections teaches one principle. Treat them as a checklist you
can run against any figure:

| # | Principle | The question it asks |
|---|---|---|
| 1 | **Context Matters** | *Compared to what?* Which comparison does your design license: a control group, a benchmark, a prior estimate? |
| 2 | **Visualization Derives From Data** | *What kind of data is this?* Which chart respects its shape and the scale of the measure? |
| 3 | **Less Is More** | *What can I delete and still understand the chart?* And what must never be deleted? |
| 4 | **Hierarchy Among Data** | *Where does the eye land first? Should it?* |
| 5 | **Beauty Counts** | *Which one number am I asking the reader to remember?* |
| 6 | **Telling Your Story** | *Can a stranger restate the bounded conclusion in one sentence?* |

One rule sits above all six, and it comes straight from EDR|AI ch. 32: **figure
honesty**, the rule that a figure's picture may never say more than its data. Three
companions travel with it. The **claim boundary** is the line between what your
evidence licenses and what it does not, carried by your headline verb.
**Uncertainty on the page** means the interval sits in the same glance as the
number. **Accessibility** means no distinction is carried by color alone. When a
principle below would break one of these four, the principle loses.

---

## 3. Context Matters

Every figure has a goal and an audience. Three rules follow.

- **Separate exploration from the finding.** Your notebook holds every chart you
  tried. The poster holds the few that carry the claim.
- **Adapt to the audience.** An Expo visitor is rarely a specialist in your field.
  They need the comparison and the boundary, not your debugging history.
- **Keep one learning objective in mind.** It tells you which information belongs
  and which is filler.

An isolated number says very little. In research, the context is the **licensed
comparison**: the comparison your design actually supports, the one that turns a
number into evidence. Example: in a randomized experiment, the licensed comparison
is the control group; in a descriptive survey, it may be a census benchmark or the
same question asked in an earlier wave.

The cell below shows the difference on the field experiment. In `foos_etal`, voters
on randomly chosen streets received a get-out-the-vote treatment and the rest did
not, and the outcome is whether each person voted. Two design facts shape the
numbers: streets were assigned with unequal probabilities, so the file's design
weights restore each voter's intended share, and streets are the unit of
assignment, so the intervals are clustered by street. Run it. Expect a turnout share
near one half that looks like a result on its own, and a treatment-minus-control
gap of a few points whose interval tells a more careful story.

In [ ]:
# One number alone, then the same number beside the comparison the design licenses.
streets = pd.factorize(foos["street"])[0]
fit = smf.wls("marked_register_2014 ~ treat", data=foos, weights=foos["weights"]).fit(
    cov_type="cluster", cov_kwds={"groups": streets})     # streets were assigned, so cluster by street

arm = {}
for g, label in [(0, "Control"), (1, "Treatment")]:
    d = foos[foos["treat"] == g]
    m = smf.wls("marked_register_2014 ~ 1", data=d, weights=d["weights"]).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(d["street"])[0]})
    arm[label] = (100 * m.params.iloc[0], 100 * 1.96 * m.bse.iloc[0])   # share and 95% half-width

effect = 100 * fit.params["treat"]
lo, hi = 100 * fit.conf_int().loc["treat"]
print(f"Treatment turnout : {arm['Treatment'][0]:.1f}% ± {arm['Treatment'][1]:.1f}")
print(f"Control turnout   : {arm['Control'][0]:.1f}% ± {arm['Control'][1]:.1f}")
print(f"Difference        : {effect:+.1f} points, 95% interval {lo:+.1f} to {hi:+.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 2]})

# LEFT: the lonely number.
axes[0].text(0.5, 0.55, f"{arm['Treatment'][0]:.0f}%", ha="center", va="center",
             fontsize=44, fontweight="bold", color=ACCENT)
axes[0].text(0.5, 0.2, "of treated voters turned out", ha="center", fontsize=11)
axes[0].axis("off")
axes[0].set_title("Without context", loc="left")

# RIGHT: the same number beside its control, with both intervals, on the full 0-100 scale.
labels = ["Control", "Treatment"]
vals = [arm[k][0] for k in labels]
errs = [arm[k][1] for k in labels]
axes[1].barh(labels, vals, xerr=errs, capsize=5, color=[GREY, ACCENT])
for i, (v, e) in enumerate(zip(vals, errs)):
    axes[1].text(v + e + 1.5, i, f"{v:.1f}%", va="center")
axes[1].set_xlim(0, 100)
axes[1].set_xlabel("Turned out to vote (%), weighted, 95% intervals clustered by street")
axes[1].set_title(f"With context: a {effect:+.1f}-point gap, 95% interval {lo:+.1f} to {hi:+.1f}",
                  loc="left")
declutter(axes[1], keep_left=True)
plt.tight_layout()
plt.show()

assert lo < 0 < hi, "the interval for the gap should cross zero in this file"
print("\n✓ One number, two readings: only the comparison tells you which one the data support.")


**Reading the output.** On its own, the left panel invites you to read 51% as an
achievement. Beside its control the picture changes: treated voters turned out at
50.7% (± 2.1) and control voters at 47.9% (± 3.3), a gap of 2.8 points whose 95%
interval runs from −1.0 to +6.7. That is a positive estimate the data cannot yet
tell apart from no effect, and only the comparison lets a reader see it. Notice the
control's interval is wider than the treatment's: fewer streets were assigned to
control, and the interval counts streets, not people.

> **A question that often comes up here:** *"What counts as a fair comparison on a
> research poster?"* The one your design licenses, and no other. An experiment
> compares with its control. A descriptive survey compares with a population
> benchmark or with a published estimate you verified in your Studio 3 evidence
> map. A prediction compares with a simple baseline on held-out cases. If your
> poster reports a number with nothing beside it, a judge will supply a comparison
> of their own, and it will rarely be the right one.

---

## 4. Visualization Derives From Data

Before you choose a chart, ask three questions about the data behind it.

1. **What type of data is it?** Categories, ordered levels, continuous values, or a
   sequence over time?
2. **How many dimensions are you encoding?** One, two, or more?
3. **Where will it be read?** A poster is read on a flat page from four feet away,
   so more than two dimensions on one panel usually confuses the reader.

And one rule overrides all three: **respect the scale of the measure.** A bad axis
can flip a figure's conclusion without changing the data.

### 4.1 Use Graphs Instead of Tables

Political scientists Jonathan Kastellec and Eduardo Leoni (2007) showed that their
field reports results in tables far more often than in graphs, even though graphs
make results clearer, and they rebuilt published tables, regression results
included, as graphs that are easier to read correctly. The cell below runs one
small regression on a project-sized subsample of the LAPOP teaching survey and
prints it both ways.

A **regression coefficient** here is the average difference in police trust (a
1-to-7 item) between respondents one standard deviation apart on one item, among
respondents similar on the other items in the model. Example: a coefficient of 0.3
on government responsiveness means respondents who rate the government one
standard deviation more responsive report, on average, 0.3 more points of police
trust. That is an **association**: nothing here was randomized, so it is not a
cause. Municipalities are the survey's clusters, so the intervals are clustered by
municipality.

Expect a printed table that technically holds every answer, then a dot plot that
lets you rank the same estimates and see which intervals cross zero in a few
seconds.

In [ ]:
# One small regression, printed as a table and then drawn as a coefficient plot.
sub = lapop.sample(n=800, random_state=SEED).copy()          # seeded, project-sized subsample
items = {"govt_responsive": "Government is responsive",
         "govt_pride": "Pride in the government",
         "support_political_system": "Support for the political system",
         "self_efficacy_political": "Political self-efficacy",
         "ideology": "Ideology (left to right)"}
for col in items:                                            # one standard deviation = one unit
    sub[col + "_z"] = (sub[col] - sub[col].mean()) / sub[col].std()
formula = "trust_police ~ " + " + ".join(c + "_z" for c in items)
ols = smf.ols(formula, data=sub).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(sub["municipality"])[0]})

ci = ols.conf_int()
table = pd.DataFrame({"estimate": ols.params, "std. error": ols.bse,
                      "95% low": ci[0], "95% high": ci[1]}).drop("Intercept")
table.index = [items[name[:-2]] for name in table.index]
print("THE TABLE (every number is here, and the pattern is hard to see):\n")
print(table.round(3).to_string())

# THE GRAPH: the same numbers, sorted, with intervals and a reference line at zero.
t = table.sort_values("estimate")
crosses = ((t["95% low"] < 0) & (t["95% high"] > 0)).to_numpy()
colors = np.where(crosses, GREY, ACCENT)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hlines(range(len(t)), t["95% low"], t["95% high"], color=colors, lw=3)
ax.scatter(t["estimate"], range(len(t)), color=colors, zorder=3, s=60)
ax.axvline(0, color=DARK, lw=1, ls="--")
ax.set_yticks(range(len(t)))
ax.set_yticklabels([f"{name}  {est:+.2f}" for name, est in zip(t.index, t["estimate"])])
ax.set_xlabel("Difference in police trust (1-7 scale) per one-standard-deviation step,\n"
              "with its 95% interval, clustered by municipality")
ax.set_title("Which items travel with police trust? Grey intervals cross zero", loc="left")
declutter(ax)
plt.tight_layout()
plt.show()
print(f"n = {int(ols.nobs)} respondents in {sub['municipality'].nunique()} municipalities "
      "(teaching resample: practice only, no claim about Brazil)")


**Reading the output.** The table holds twenty numbers, and finding which items
matter means reading all of them. The plot sorts the same estimates in a glance:
support for the political system (+0.47), pride in the government (+0.28), and
government responsiveness (+0.24) travel with police trust, with intervals clear of
zero, while ideology (+0.04) and political self-efficacy (−0.16) have intervals that
cross zero. Self-efficacy's interval only just reaches zero (−0.33 to +0.005), a
reminder that "crosses zero" is a line drawn on a continuum, not a verdict. Every
estimate is an association, in a teaching resample.

> **A question that often comes up here:** *"So when does a table earn its place?"*
> When the reader must look up an exact value rather than see a pattern: sample
> sizes by group, or the full model behind your headline figure. On a poster that
> is almost never the reader's job. Put the complete table behind your QR code and
> keep the graph on the wall.

---

### 4.2 Scales Can Mislead

You met the worst offender in nb10: the **truncated axis**, an axis that starts
above the bottom of its scale. Floored at 4.26, a 0.51-point gap on a 1-to-7 trust
scale was drawn 11.3 times taller than it is. Nothing was rounded or invented; the
axis alone changed the finding.

The rule from EDR|AI ch. 32: run the axis across **the full scale of the measure**.
For counts, shares, and rates, that means starting at zero. For a bounded scale such
as 1 to 7, it means 1 to 7. When a narrower window is genuinely justified, as for a
slow-moving series over time, say so in the caption, in words a stranger reads
before the picture.

---

### 4.3 Avoid Dual Axes, and Never Reach for a Third

A **dual-axis chart** puts two series with different units on one panel, one read
on the left axis and one on the right. Example: survey invitations sent each week
on the left, the response rate on the right. The trouble is that whoever picks the
two axis ranges controls how the lines line up, so the apparent relationship
belongs to the designer, not to the data.

The cell below shows it with one simulated fieldwork log (seed 464). It draws the
same two series twice, changing only the range of the right-hand axis, and then
draws the honest version: two panels that share the week axis. Expect the first two
panels to tell different stories about identical numbers.

In [ ]:
# A simulated fieldwork log (seed 464): invitations sent and the response rate.
sim = np.random.default_rng(SEED)
weeks = np.arange(1, 13)
invites = np.round(400 + 35 * weeks + sim.normal(0, 25, weeks.size))
rate = 100 * np.clip(0.31 - 0.004 * weeks + sim.normal(0, 0.012, weeks.size), 0, 1)
print(f"Response rate: {rate[0]:.1f}% in week 1, {rate[-1]:.1f}% in week 12")

fig = plt.figure(figsize=(15, 4.6))
gs = fig.add_gridspec(2, 3)
for col, (lo_r, hi_r), title in [(0, (0, 40), "Right axis 0-40: the rate looks steady"),
                                 (1, (24, 33), "Right axis 24-33: the rate looks like it collapses")]:
    ax = fig.add_subplot(gs[:, col])
    ax.plot(weeks, invites, color=DARK, lw=2)
    ax.set_ylim(0, 1000)
    ax.set_ylabel("Invitations sent (solid)")
    ax.set_xlabel("Week of fieldwork")
    ax2 = ax.twinx()
    ax2.plot(weeks, rate, color=ACCENT, lw=2, ls="--")
    ax2.set_ylim(lo_r, hi_r)
    ax2.set_ylabel("Response rate, % (dashed)", color=ACCENT)
    ax.set_title(title, loc="left", fontsize=10)

# THE HONEST VERSION: two panels, one shared week axis, each on its own full scale.
top = fig.add_subplot(gs[0, 2])
bottom = fig.add_subplot(gs[1, 2], sharex=top)
top.plot(weeks, invites, color=DARK, lw=2)
top.set_ylim(0, 1000)
top.set_title("Two panels, one shared x-axis", loc="left", fontsize=10)
top.text(12.3, invites[-1], "Invitations", va="center", fontsize=9)
bottom.plot(weeks, rate, color=ACCENT, lw=2)
bottom.set_ylim(0, 40)
bottom.text(12.3, rate[-1], "Response rate, %", va="center", fontsize=9, color=ACCENT)
bottom.set_xlabel("Week of fieldwork")
for a in (top, bottom):
    declutter(a, keep_left=True)
    a.set_xlim(0.5, 15)
    a.set_xticks(range(2, 13, 2))
plt.tight_layout()
plt.show()


**Reading the output.** The rate fell by about four points over twelve weeks,
from roughly 30% to roughly 26%. On the left, with a 0-to-40 right axis, that fall
looks like a gentle drift beside rising invitations. In the middle, with a 24-to-33
right axis, the same numbers look like a collapse. In both panels the lines cross,
and the crossing point means nothing at all: it is where two arbitrary ranges happen
to meet, and it moves whenever either range does. On the right, each series sits on its own honest scale and the two panels
share only what they truly share, the week.

If you ever reach for a **third** axis, one chart is trying to tell three stories.
Split it into three panels.

> **A question that often comes up here:** *"But I want the reader to see that two
> series move together."* Then compute the relationship and show it directly: a
> scatter plot of one against the other, or both series rescaled to the same
> starting value (for example, week 1 = 100). Either way, the reader sees the data's
> relationship instead of the designer's.

---

### 4.4 Fewer Pie Charts, and None for Ordered Data

The pie is the default "show me a breakdown" chart, and it is usually the wrong one.
The reason is perceptual: people judge lengths along a common baseline far more
accurately than angles or areas. A pie has a second problem with research data: it
has no order. The cell below draws the ideology item from the LAPOP teaching survey,
a 1-to-10 scale from left to right, as a pie and as bars in scale order. Expect to
find the most common position quickly in the bars and slowly in the pie, and to lose
the left-to-right shape entirely in the pie.

In [ ]:
# The same ten shares, drawn as a pie and as bars in the scale's own order.
shares = lapop["ideology"].value_counts(normalize=True).sort_index() * 100
mode = int(shares.idxmax())
print(f"Most common position: {mode} ({shares.max():.1f}% of the file)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
axes[0].pie(shares, labels=shares.index, colors=plt.cm.tab10.colors, startangle=90,
            counterclock=False, textprops={"fontsize": 9})
axes[0].set_title("Ideology, 1 (left) to 10 (right): the pie", loc="left")

bars = axes[1].bar(shares.index, shares, color=[ACCENT if i == mode else GREY for i in shares.index])
for x, s in zip(shares.index, shares):
    axes[1].text(x, s + 0.6, f"{s:.0f}%", ha="center", fontsize=9,
                 fontweight="bold" if x == mode else "normal")
axes[1].set_xticks(shares.index)
axes[1].set_xlabel("Ideology, 1 (left) to 10 (right)")
axes[1].set_ylim(0, shares.max() + 5)
axes[1].set_yticks([])
axes[1].set_title("The same shares in scale order", loc="left")
declutter(axes[1])
plt.tight_layout()
plt.show()


**Reading the output.** The bars show a shape the pie hides: two peaks, one at the
center of the scale (5, about 20% of the file) and one at the far right (10, about
18%). In the pie, 5 and 10 are two slices of similar size with nothing to say that one
is the center of the scale and the other its end, and comparing them means comparing
two angles. A two-peaked distribution is a finding in itself, and only the ordered
bars let a reader see it.

Rendering can also distort a chart that was otherwise defensible. A
three-dimensional tilt enlarges the slices nearest the viewer, so a 3D pie can
reorder two nearly equal shares without changing a single number. A two-dimensional
comparison never needs a third dimension.

> **A question that often comes up here:** *"Is the pie chart banned?"* No. Reserve
> it for two or three parts of a meaningful whole, when the split itself is the
> headline (for example, the share of your sample that consented to a follow-up).
> For anything ordered, or anything with more than three parts, bars in a
> meaningful order will serve your reader better.

---

### 4.5 Borrow Structure From the Galleries

Before you build a figure from scratch, browse a gallery. Each is organized by chart
type and shows the code that drew every example.

- [The Python Graph Gallery](https://python-graph-gallery.com/), chart ideas with Python code.
- [The R Graph Gallery](https://r-graph-gallery.com/), the same idea for R.
- [The Matplotlib examples gallery](https://matplotlib.org/stable/gallery/index.html),
  the library this notebook draws with.

Copy the **structure** (the encoding, the layout, the annotation strategy), then
redraw it with your own data and labels. If your AI adapts gallery code for you,
treat the result like any delegated output: run it, read the figure against your
own printed numbers, and log it.

---

## 5. Less Is More: Separate Signal From Noise

Every line, label, gridline, and decoration on a chart is a small tax on the
reader's attention. Your job is to lower that tax.

Edward Tufte named the idea the **data-ink ratio**: the share of a graphic's ink
that encodes data.

$$\text{Data-ink ratio} = \frac{\text{ink that encodes data}}{\text{total ink used to print the graphic}}$$

Example: a box drawn around a plot encodes nothing, so deleting it raises the ratio
without losing a single number. You push the ratio up by deleting ink that encodes
nothing, and only that ink.

**One EDR|AI amendment.** An error bar, a sample size, a reference line at zero, and
the caption that states your boundary are all **data-ink**: they encode what the
evidence can and cannot support. Cleanup that deletes the uncertainty is not
cleanup. It is a change of claim.

### 5.1 One Chart, Seven Cleanup Steps

The cell below walks one figure through seven cleanup steps, one decision per
panel. The data are a simulated fieldwork log (seed 464): survey responses received
each week, and the responses that passed the attention check, a question built in to
catch respondents who answer without reading. Watch what each step deletes, and ask
whether anything **a reader needs** was lost.

| Panel | What changes | Why |
|---|---|---|
| 1 | Nothing: the default chart | Frame, heavy grid, markers, long tick labels, a boxed legend, two loud colors |
| 2 | The frame goes | The axes already mark the edge |
| 3 | The heavy grid becomes faint | A poster shows a trend, not exact read-offs |
| 4 | The markers go | When the line is the message, the dots repeat it |
| 5 | Short ticks, no tilt | Week numbers say the same thing in less ink |
| 6 | The unit moves into the title | The reader's eye stays next to the data |
| 7 | Direct labels replace the legend | Each line is named where it ends |
| 8 | One quiet grey and one accent | The accent goes on the series the claim is about |

Expect eight small panels whose numbers never change while the signal gets
louder.

In [ ]:
# A simulated fieldwork log (seed 464): responses received and responses usable.
log = np.random.default_rng(SEED + 10)
wk = np.arange(1, 13)
received = np.round(180 + 40 * np.sin(wk / 2) + log.normal(0, 12, wk.size))
usable = np.round(received * np.clip(0.86 - 0.012 * wk + log.normal(0, 0.02, wk.size), 0, 1))

def draw_step(ax, step):
    """Draw the fieldwork chart as it looks after cleanup panel `step` (1 to 8)."""
    c_rec, c_use = ("#1f77b4", "#d62728") if step < 8 else (GREY, ACCENT)
    mk_rec = dict(marker="o", ms=5) if step < 4 else {}
    mk_use = dict(marker="s", ms=5) if step < 4 else {}
    ax.plot(wk, received, color=c_rec, lw=2, label="Responses received", **mk_rec)
    ax.plot(wk, usable, color=c_use, lw=2.6 if step >= 8 else 2,
            label="Responses passing the attention check", **mk_use)
    ax.set_ylim(0, 260)
    if step >= 2:                                    # 2: the frame goes
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    if step < 3:                                     # 3: the heavy grid becomes faint
        ax.grid(True, color="#666666", lw=1)
    else:
        ax.grid(axis="y", color="#E5E5E5", lw=0.8)
        ax.set_axisbelow(True)
    ax.set_xticks(wk)
    if step < 5:                                     # 5: short ticks, no tilt
        ax.set_xticklabels([f"Fieldwork week {w}" for w in wk], rotation=45, ha="right", fontsize=6)
    else:
        ax.set_xlabel("Week of fieldwork", fontsize=8)
        ax.spines["left"].set_visible(False)
        ax.tick_params(length=0)
    if step < 6:                                     # 6: the unit moves into the title
        ax.set_ylabel("Count of survey responses (n)", fontsize=8, labelpad=14)
        ax.set_title("Responses", fontsize=9)
    else:
        ax.set_title("Survey responses per week", fontsize=9, loc="left")
    if step < 7:                                     # 7: direct labels replace the legend
        ax.legend(fontsize=6, loc="lower center", frameon=True, edgecolor="black")
    else:
        ax.set_xlim(0.5, 15)
        ax.text(12.3, received[-1], "Received", color=DARK if step >= 8 else c_rec,
                fontsize=8, va="center")
        ax.text(12.3, usable[-1], "Usable", color=c_use, fontsize=8, va="center",
                fontweight="bold" if step >= 8 else "normal")

names = ["1 Default", "2 No frame", "3 Faint grid", "4 No markers",
         "5 Short ticks", "6 Unit in title", "7 Direct labels", "8 Grey + one accent"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7.4))
for step, (ax, name) in enumerate(zip(axes.flat, names), start=1):
    draw_step(ax, step)
    ax.text(0, 1.2, name, transform=ax.transAxes, fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Same 24 numbers in every panel: received {int(received.sum())}, usable {int(usable.sum())}.")


**Reading the output.** Every panel draws the same 24 numbers. The cleanup did
three things, in order: it deleted ink that encoded nothing (the frame and the heavy
grid), it replaced ink that repeated itself (markers on a line, a legend that sent
the eye away from the data), and it made the remaining ink earn its place (short
ticks, the unit in the title, one accent). Nothing a reader needs was lost, and
nothing that carries uncertainty was on this chart to lose. On your own figures,
that last check is the one that matters most.

### 5.2 Before and After

Run the cell below to see the first and last panels at full size, side by side.

In [ ]:
# Before and after, at a size closer to how a poster reader meets them.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
draw_step(axes[0], 1)
draw_step(axes[1], 8)
axes[0].text(0, 1.12, "Before", transform=axes[0].transAxes, fontsize=12, fontweight="bold")
axes[1].text(0, 1.12, "After", transform=axes[1].transAxes, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


> **A question that often comes up here:** *"Isn't a stripped-down chart cold or
> boring?"* It is the opposite. The more non-data ink you remove, the louder the
> remaining ink becomes. The right-hand panel does not lose information; it
> amplifies the signal that was already there.

> 🤔 **Quick check, your own figure:** open the figure that carries your lead claim.
> Name two pieces of non-data ink you could delete without losing information, and
> one piece of uncertainty ink you must keep. Hold the answer; Exercise 1 asks for it
> formally.

---

## 6. Hierarchy Among Data

Once a chart is clean, you choose **what the reader looks at first**. That choice is
**visual hierarchy**: the order in which the eye visits the elements of a figure or a
page. Example: on a poster seen from four feet, the headline number should be the
first thing the eye lands on. The eye is not democratic; it lands on whatever is
loudest. Your job is to make the loudest thing the right thing.

### 6.1 Count the 3s

Run the cell below, then count how many times the digit 3 appears. Time yourself.
Expect it to take ten seconds or more, and expect not to be sure of your answer.

In [ ]:
# Count the 3s. Count before you run the next cell.
digits = np.random.default_rng(SEED).integers(0, 10, size=(4, 12))

def digit_grid(highlight):
    fig, ax = plt.subplots(figsize=(11, 2.8))
    for r in range(digits.shape[0]):
        for c in range(digits.shape[1]):
            is_three = highlight and digits[r, c] == 3
            ax.text(c, -r, str(digits[r, c]), fontsize=26, ha="center", va="center",
                    color=DARK if is_three else "#BBBBBB",
                    fontweight="bold" if is_three else "normal")
    ax.set_xlim(-0.6, 11.6)
    ax.set_ylim(-3.6, 0.6)
    ax.axis("off")
    plt.show()

digit_grid(highlight=False)


### 6.2 Now Try Again, With Hierarchy

Same digits, one visual change. Expect the count to take two seconds.

In [ ]:
digit_grid(highlight=True)
print(f"Answer: {int((digits == 3).sum())} threes.")


**Reading the output.** The task collapsed from a search to a glance because one
property changed: the 3s became darker and heavier than everything around them.
Properties like darkness and weight are **preattentive attributes**: visual
features the eye registers before you consciously look, in a fraction of a second.
Example: one dark digit among grey ones. The reader will look at the loudest thing
on your figure whether you designed it that way or not. You are not choosing
whether hierarchy exists. You are choosing whether you control it.

### 6.3 Six Dials That Win the Eye

You have six dials to turn: **size**, **weight**, **color**, **intensity**,
**position**, and **enclosure**. The cell below shows each one. Expect one element
in every panel to pull your eye before you decide to look at it.

In [ ]:
# Six preattentive dials, one panel each: one element differs, the rest are context.
xs, ys = np.meshgrid(np.arange(6), np.arange(3))
xs, ys = xs.ravel(), ys.ravel()
fig, axes = plt.subplots(2, 3, figsize=(13, 5.4))
for ax, dial in zip(axes.flat, ["Size", "Weight", "Color", "Intensity", "Position", "Enclosure"]):
    if dial == "Weight":                       # lines instead of dots: one is heavier
        for i in range(6):
            ax.plot([i, i], [0, 2], color=DARK, lw=6 if i == 3 else 1.2)
    else:
        size = np.full(xs.size, 60.0)
        color = np.array([GREY] * xs.size, dtype=object)
        y = ys.astype(float).copy()
        if dial == "Size":
            size[8] = 330
        if dial == "Color":
            color[8] = ACCENT
        if dial == "Intensity":
            color[:] = "#DDDDDD"
            color[8] = DARK
        if dial == "Position":
            y[8] += 0.45
        ax.scatter(xs, y, s=size, c=list(color))
        if dial == "Enclosure":
            ax.add_patch(plt.Rectangle((xs[8] - 0.35, ys[8] - 0.35), 0.7, 0.7,
                                       fill=False, ec=DARK, lw=1.5))
    ax.set_xlim(-0.7, 5.7)
    ax.set_ylim(-0.7, 2.9)
    ax.axis("off")
    ax.set_title(dial, loc="left", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### 6.4 Highlighting With Color, Backed by a Second Channel

Of the six dials, color is the one you will reach for most often on a poster: it is
the cheapest hierarchy available. It is also the one a reader can lose: about 8% of
men, roughly 1 in 12, have some color-vision deficiency (Crameri, Shephard, and
Heron, 2020), so EDR|AI ch. 32 asks for
**redundant encoding**: carrying a distinction in more than one channel, so color is
never the only signal. Example: the accented item below is also labeled in bold,
with its value printed beside it.

The cell draws mean ratings of five institutions and attitudes from a seeded
subsample of 600 LAPOP respondents, each on its full 1-to-7 scale with a 95%
interval. Suppose your claim is about the police. Expect the police row to win the
eye three ways at once: color, weight, and a printed value.

In [ ]:
# Five ratings on one 1-7 ruler; the accent is spent on the one the claim is about.
s600 = lapop.sample(n=600, random_state=SEED + 2)
ratings = {"trust_police": "Trust in the police",
           "trust_military": "Trust in the military",
           "trust_supreme_court": "Trust in the Supreme Court",
           "support_political_system": "Support for the political system",
           "govt_pride": "Pride in the government"}
summary = pd.DataFrame({
    "mean": [s600[c].mean() for c in ratings],
    "half": [1.96 * s600[c].std(ddof=1) / np.sqrt(len(s600)) for c in ratings]},
    index=list(ratings.values())).sort_values("mean")
print(summary.to_string(float_format=lambda v: f"{v:.2f}"))

fig, ax = plt.subplots(figsize=(9, 3.8))
for i, (name, row) in enumerate(summary.iterrows()):
    focus = name == "Trust in the police"
    col = ACCENT if focus else GREY
    ax.hlines(i, row["mean"] - row["half"], row["mean"] + row["half"], color=col, lw=3)
    ax.scatter(row["mean"], i, color=col, s=70 if focus else 40, zorder=3)
    ax.text(row["mean"] + row["half"] + 0.08, i, f"{row['mean']:.2f}", va="center",
            fontsize=10, color=col, fontweight="bold" if focus else "normal")
ax.set_yticks(range(len(summary)))
ax.set_yticklabels(summary.index)
for lab in ax.get_yticklabels():
    if lab.get_text() == "Trust in the police":
        lab.set_fontweight("bold")
ax.set_xlim(1, 7)
ax.set_xlabel("Mean rating, 1 (none) to 7 (a lot), with 95% intervals")
ax.set_title("Where the police sit among five ratings (teaching resample, n = 600)", loc="left")
declutter(ax)
plt.tight_layout()
plt.show()


**Reading the output.** The police row wins the eye three ways at once: the accent
color, the bold label, and the printed value, 4.51. Print it in grayscale and the bold
label and the value still mark it, so the message survives a color-blind reader and a
black-and-white printer. The other four rows give context without competing: the
military rates highest (5.13) and pride in the government lowest (2.98), all on the
same 1-to-7 ruler, each with an interval of about ±0.16 to ±0.17.

> **A question that often comes up here:** *"How many colors should a poster figure
> have?"* For a headline figure, two: one quiet base color for everything that is
> context, and **one** accent for the element you want remembered. If a chart uses
> more than three colors, ask whether each one does structural work, and grey out
> the ones that do not. Then print the figure in grayscale. If the message survives,
> your accent is backed by a second channel. If it vanishes, it was carried by color
> alone.

### 6.5 Returning to the Cleaned Chart

The cleaned fieldwork chart from Section 5 is accurate and calm, but it does not
yet tell the reader what to notice. The cell below adds one layer of hierarchy: an
annotation at the week where the share of usable responses was lowest. Expect the
eye to go to that week first.

In [ ]:
# Add one annotation to the cleaned chart: the week with the lowest usable share.
share = usable / received
worst = int(np.argmin(share))
fig, ax = plt.subplots(figsize=(10, 4.6))
draw_step(ax, 8)
ax.annotate(f"Week {wk[worst]}: {int(usable[worst])} of {int(received[worst])} usable "
            f"({share[worst]:.0%})",
            xy=(wk[worst], usable[worst]), xytext=(wk[worst] - 5.5, 40),
            arrowprops=dict(arrowstyle="->", color=ACCENT), color=ACCENT, fontsize=10)
plt.tight_layout()
plt.show()
print(f"Usable share across the 12 weeks: {share.min():.0%} at the lowest, {share.max():.0%} at the highest.")


> 🤔 **Quick check, where does the eye land?** Open your headline figure. Where
> does your eye land first? Is it the element your bounded sentence is about? If
> not, which single change (size, weight, color backed by a label, or one
> annotation) would move it?

---

## 7. Beauty Counts: Not All Data Are Equally Important

Hierarchy decides what is loudest. Beauty makes the chart pleasant enough that the
reader **stays** long enough to hear the message. The two work together. A chart
with clear hierarchy but a jarring look loses readers before the data registers. A
polished chart with no hierarchy buries its point; the reader enjoys the look and
walks away without a number.

Beauty here is not decoration. It is alignment, whitespace, a readable typeface,
and a calm palette, the choices that tell a reader your work was done with care
before they read a single number. On an Expo floor, that signal can earn the extra
fifteen seconds that turn a glance into a conversation.

The cell below draws one comparison two ways: how respondents answered the police
trust item (1 to 7), split by how responsive they rated the government, using
nb10's groups and the whole teaching file. On the left, every answer gets its own
bright color. On the right, the answers 5 to 7, which together mean trusting the
police, carry the accent and a printed share, and everything else recedes. Expect
to compare seven colors on the left and two numbers on the right.

In [ ]:
# One comparison, two designs: seven loud segments versus one emphasized share.
groups = {"Rated government 1-3": lapop[lapop["govt_responsive"] <= 3],
          "Rated government 4-7": lapop[lapop["govt_responsive"] >= 4]}
dist = pd.DataFrame({g: d["trust_police"].value_counts(normalize=True).sort_index() * 100
                     for g, d in groups.items()}).T
trusting = dist[[5, 6, 7]].sum(axis=1)
print("Share answering 5, 6, or 7 on police trust:")
for g, v in trusting.items():
    print(f"  {g}: {v:.0f}%   (n = {len(groups[g]):,} rows in the file)")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
rainbow = plt.cm.tab10.colors
for ax, emphasize in zip(axes, [False, True]):
    left = np.zeros(len(dist))
    if emphasize:          # answers 1-4 as quiet greys, then 5-7 merged into one accent
        segments = [(dist[k], ["#EEEEEE", "#E2E2E2", "#D6D6D6", "#CACACA"][k - 1]) for k in (1, 2, 3, 4)]
        segments.append((trusting, ACCENT))
    else:
        segments = [(dist[k], rainbow[k - 1]) for k in dist.columns]
    for j, (width_k, col) in enumerate(segments):
        ax.barh(dist.index, width_k, left=left, color=col, edgecolor="white",
                label=None if emphasize else str(j + 1))
        left += width_k.to_numpy()
    ax.set_xlim(0, 100)
    declutter(ax)
    ax.set_xticks([])
    if emphasize:
        for i, v in enumerate(trusting):
            ax.text(100 - v / 2, i, f"{v:.0f}% trust\n(5 to 7)", ha="center", va="center",
                    color="white", fontsize=10, fontweight="bold")
        ax.set_title("Emphasis: the share that trusts the police", loc="left")
    else:
        ax.legend(title="Police trust answer", ncol=7, fontsize=8, title_fontsize=8,
                  loc="upper center", bbox_to_anchor=(0.5, -0.05), frameon=False)
        ax.set_title("Every answer in its own color", loc="left")
plt.tight_layout()
plt.show()


**Reading the output.** Both panels draw identical shares. On the left, comparing
the two groups means comparing seven colored segments, and nothing tells the eye
which ones matter. On the right, the greys recede and one number per group carries
the comparison: 45% of respondents who rated the government 1 to 3 answered 5 or
higher on police trust, against 63% of those who rated it 4 to 7. Two cautions travel
with the emphasis. Where "trusting" begins is a measurement decision, so state it on
the figure, as here, and keep it the same everywhere on the board. And these shares
describe the teaching file, not Brazil.

> **A question that often comes up here:** *"Can I use Purdue's colors on my
> poster?"* Yes. Old Gold and black make a strong one-accent palette. The rule is
> not "avoid brand colors" but "spend the accent on one element." If every bar,
> heading, and border is gold, nothing stands out. Let one element (the headline
> figure's key series, or the title banner) carry the brand, and keep the rest
> neutral.

> 🤔 **Quick check, the one-number test:** what is the single number your poster
> asks a reader to remember? Is there a figure in your work that makes that number
> louder than everything around it, with its interval beside it? If not, which
> mark would you spend the accent on?

---

## 8. Telling Your Story

The last principle ties the other five together. A clean chart with good hierarchy
and a calm palette still has to tell a **specific** story: a question, an answer,
and a boundary the reader walks away with. In evidence-driven research, the story
has one more constraint. Its last line is your **bounded headline**, the sentence
whose verb your compass position allows.

The cell below walks one figure through five steps, drawn from a seeded subsample
of 1,200 LAPOP respondents: mean trust in three institutions at each level of rated
government responsiveness, 1 to 7. Expect the first panel to say nothing in
particular and the last one to say exactly one bounded thing.

In [ ]:
# One figure, five steps: from a default chart to a bounded headline.
s1200 = lapop.sample(n=1200, random_state=SEED + 1)
inst = {"trust_police": "Police", "trust_military": "Military", "trust_supreme_court": "Supreme Court"}
by = s1200.groupby("govt_responsive")
means = by[list(inst)].mean()
half = 1.96 * by[list(inst)].std(ddof=1) / np.sqrt(by.size().to_numpy()[:, None])
counts = by.size()
levels = means.index.to_numpy()
print("Respondents at each responsiveness level:", {int(k): int(v) for k, v in counts.items()})
p1, p7 = means.loc[1, "trust_police"], means.loc[7, "trust_police"]
print(f"Mean police trust: {p1:.2f} at level 1, {p7:.2f} at level 7 "
      f"(± {half.loc[1, 'trust_police']:.2f} and ± {half.loc[7, 'trust_police']:.2f})")

fig, axes = plt.subplots(2, 2, figsize=(14, 8.6))
width = 0.27
# Step 1: the default chart: grouped bars, rainbow, legend, no intervals, a vague title.
ax = axes[0, 0]
for j, (col, name) in enumerate(inst.items()):
    ax.bar(levels + (j - 1) * width, means[col], width, color=plt.cm.tab10.colors[j + 1], label=name)
ax.legend(fontsize=8)
ax.set_title("Step 1. Trust by responsiveness", fontsize=11)
# Step 2: clean: no frame, one grey, the legend kept for now.
ax = axes[0, 1]
for j, (col, name) in enumerate(inst.items()):
    ax.bar(levels + (j - 1) * width, means[col], width, color=["#CCCCCC", "#999999", "#666666"][j],
           label=name)
ax.legend(fontsize=8, frameon=False)
declutter(ax, keep_left=True)
ax.set_title("Step 2. Cleaned", loc="left", fontsize=11)
# Step 3: highlight the series the claim is about.
ax = axes[1, 0]
for j, (col, name) in enumerate(inst.items()):
    ax.bar(levels + (j - 1) * width, means[col], width, color=ACCENT if col == "trust_police" else "#DDDDDD")
ax.text(0.02, 0.95, "Police", color=ACCENT, fontweight="bold", transform=ax.transAxes)
declutter(ax, keep_left=True)
ax.set_title("Step 3. The police highlighted", loc="left", fontsize=11)
# Step 4: the right chart for an ordered scale: points and lines, with intervals.
ax = axes[1, 1]
for col, name in inst.items():
    if col != "trust_police":
        ax.plot(levels, means[col], color="#CCCCCC", lw=1.5)
        ax.text(7.15, means.loc[7, col], name, color="#999999", fontsize=8, va="center")
ax.errorbar(levels, means["trust_police"], yerr=half["trust_police"], color=ACCENT, lw=2.5,
            marker="o", capsize=3)
ax.text(7.15, means.loc[7, "trust_police"], "Police", color=ACCENT, fontweight="bold", va="center")
declutter(ax, keep_left=True)
ax.set_title("Step 4. An ordered scale gets a line, with 95% intervals", loc="left", fontsize=11)
for ax in axes.flat:
    ax.set_ylim(1, 7)
    ax.set_xticks(levels)
    ax.set_xlabel("Rated government responsiveness (1-7)", fontsize=9)
axes[1, 1].set_xlim(0.6, 8.2)
plt.tight_layout()
plt.show()


**Step 5, the final narrative.** The last step replaces the vague title with the
bounded headline and puts the boundary where the reader will meet it: directly under
the headline, in the same glance. Expect a title that states a finding, a subtitle
that states its limits, and one series that carries the accent.

In [ ]:
# Step 5: the headline states the bounded finding; the subtitle states its boundary.
fig, ax = plt.subplots(figsize=(11, 5.4))
for col, name in inst.items():
    if col != "trust_police":
        ax.plot(levels, means[col], color="#D0D0D0", lw=1.5)
        ax.text(7.15, means.loc[7, col], name, color="#9A9A9A", fontsize=9, va="center")
ax.errorbar(levels, means["trust_police"], yerr=half["trust_police"], color=ACCENT, lw=2.8,
            marker="o", capsize=3)
for x in (1, 7):
    ax.text(x, means.loc[x, "trust_police"] + half.loc[x, "trust_police"] + 0.2,
            f"{means.loc[x, 'trust_police']:.1f}", ha="center", color=ACCENT, fontweight="bold")
ax.text(7.15, means.loc[7, "trust_police"], "Police", color=ACCENT, fontweight="bold", va="center")
ax.set_ylim(1, 7)
ax.set_xlim(0.6, 8.3)
ax.set_xticks(levels)
ax.set_xlabel("How responsive the respondent rates the government (1 = not at all, 7 = very)")
ax.set_ylabel("Mean trust (1-7)")
fig.suptitle(f"Respondents who rate the government more responsive report more trust in the police:\n"
             f"{p1:.1f} at the bottom of the scale, {p7:.1f} at the top",
             x=0.02, ha="left", fontsize=12, fontweight="bold")
ax.set_title("An association in a teaching resample (n = 1,200), not an effect and not a claim "
             "about Brazil. 95% intervals treat respondents as independent.",
             loc="left", fontsize=9, color=DARK)
declutter(ax, keep_left=True)
plt.tight_layout()
plt.show()


**Reading the output.** Step 1 shows 21 bars and answers no question in particular.
Step 2 removes the noise, Step 3 tells the eye which series matters, and Step 4 picks
the right chart for an ordered scale: a line, with intervals that widen at levels 6
and 7, where fewer respondents sit (86 and 107, against 306 at level 1). The final
figure states the finding in its headline: mean police trust rises from 3.8 at the
bottom of the responsiveness scale to 5.1 at the top. Its subtitle states the
boundary in the same glance: an association in a teaching resample, not an effect and
not a claim about Brazil. The intervals treat respondents as independent. They share
municipalities, so a real analysis would cluster by municipality, and the intervals
would widen.

> **A question that often comes up here:** *"Why not just title it 'Responsive
> government builds police trust'? It is shorter."* Because "builds" is a causal
> verb, and this comparison randomized nothing: people who trust institutions in
> general may both rate the government kindly and trust the police. The descriptive
> verb "report" is what this design licenses. A judge who knows research reads the
> verb before the figure, and a causal verb over an observational comparison is the
> overreach that costs you the whole board.

---

## 📝 Exercise 1: Audit One Figure From Your Project

**Task:** pick the one figure that carries your lead claim, the headline figure of
your M9 bounded note or of your M10 artifact draft. Audit it against the six
principles and the four rules above them.

**Commit first, before any AI.** In the answer cell, write the one sentence this
figure must deliver: your bounded headline, with its verb, its reach, and its
uncertainty. Every audit judgment is measured against that sentence, so it has to
exist before a tool sees your figure.

**Then audit, one sentence per line:**

1. **Context:** what comparison is on the chart, and is it the one your design licenses?
2. **Visualization:** does the chart type respect the data's shape and the full scale of the measure?
3. **Less is more:** what ink could go without losing information, and is the uncertainty still there afterward?
4. **Hierarchy:** where does the eye land first, and should it?
5. **Beauty:** how many colors are there, and is the accent backed by a second channel?
6. **Story:** cover the title and caption; could a stranger restate your bounded sentence from the figure?

Finish with a **rebuild plan** of three changes, each small enough to make in ten
minutes.

> 💡 **AI Prompt:** "I am auditing the headline figure of my research poster. It is
> a [chart type] showing [variables] for [the comparison]. My bounded headline is:
> '[paste your committed sentence]'. Walk me through six principles of data
> communication (context, visualization, less is more, hierarchy, beauty, story) and
> name, for each, the single most likely problem in a figure like this. Then list
> every element of the figure that could make a reader conclude something bigger,
> more certain, or more causal than my headline. Do not rewrite my headline or
> redesign the figure."
>
> **After running, verify (counters *illusion of completeness*: a tidy six-part
> answer that never mentions the axis, the interval, or the color channel):**
> - [ ] Each principle gets a concrete suggestion about **your** figure, not generic
>   advice such as "make it clearer."
> - [ ] Check every suggestion against the figure you can see. Advice about a legend
>   your figure does not have is invented.
> - [ ] At least one item names the axis range, one names the uncertainty, and one
>   names a color-only distinction. If one is missing, that gap is your finding.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification,
>   with your accept-or-reject call on each suggestion.

### 🔁 Modify the Prompt

The first answer is usually generic. Rerun it with one change you choose and write
down before you run it: paste the figure's **code** instead of a description, and
ask only for "every place this code encodes a meaningful distinction with color
alone, and what a red-green color-blind reader and a screen-reader user would fail
to receive." Predict, in one line, what it will find. Then compare the prediction
with what came back.

### 🔬 Interrogate the Output

Map each suggestion the tool returned to exactly one of the six principles or one of
the four rules (figure honesty, claim boundary, uncertainty, accessibility). A
suggestion that fits nothing is noise. A principle or rule with no suggestion is the
gap in the tool's list, and naming that gap is your finding. Then reject any
suggestion that contradicts a number your own notebook printed.

### YOUR ANSWER HERE:

**My committed sentence (written before any AI):**

**Figure I am auditing:** *(name, plus a pasted image or a link)*

**1. Context:**

**2. Visualization:**

**3. Less is more:**

**4. Hierarchy:**

**5. Beauty:**

**6. Story:**

**My prediction for the modified prompt, and what it actually found:**

**The gap in the tool's list (a principle or rule with no suggestion):**

**Rebuild plan (three changes):**

1.
2.
3.

---

## 9. Why a Poster?

The principles above apply to any research artifact: a note, a slide, a figure in a
paper. From here on, the focus narrows to one artifact: your poster for the Purdue
Fall Undergraduate Research Expo. A research poster does five jobs at once.

- It **states one bounded claim** in a single static frame any visitor can scan.
- It **shows the design and the evidence** that license the claim, in the order a
  skeptic checks them.
- It **keeps the uncertainty and the boundary visible**, in the same glance as the
  claim.
- It **starts a conversation**: the board opens the discussion, and you carry the
  rest.
- It **shows that you direct the research**, including the AI you used, because
  every element on it is yours to explain and defend.

The first three are about the artifact. The last two are about you, standing next
to it. A visitor cannot scroll back, cannot ask you to clarify while you are talking
with someone else, and will not stay if the first thirty seconds fail.

> **A question that often comes up here:** *"Why a poster instead of a paper or a
> talk?"* Because it forces every constraint this notebook trains: one page, four
> feet, ninety seconds, and a board that must defend itself when you step away. A
> paper lets you hide behind length, and a talk lets you hide behind narration. A
> poster makes you commit to one claim and one figure, in public.

---

## 10. Designing an Evidence-Driven Research Poster: Objectives

Before you place a single block, lock three things, in this order.

1. **The bounded answer.** The one sentence a visitor should leave with: your
   **headline claim**, its verb chosen from your compass position, and its
   uncertainty. Example: "In this sample, respondents who rate the government more
   responsive report about half a point more trust in the police, and the two 95%
   intervals overlap."
2. **The license.** The design that earns that sentence: the pathway, the data, the
   licensed comparison, and the crossing it does or does not pay for (sampling for a
   population, assignment for a cause, a held-out check for unseen cases).
3. **The boundary and the contribution.** What the answer does not show, and what it
   adds to what was already known.

The order is the order a skeptical reader checks evidence: *what did you find, why
should I believe it, and where does it stop?* A poster that opens with methods loses
the visitor who does not yet know why to care. A poster that states a big
implication before showing the evidence loses the judge. If any of the three is
still unclear to you, write it into your claim-evidence table first. The poster
assumes those answers already exist.

---

## 11. Planning Your Poster: Template, Print Spec, and Rubric

**What the course provides.** The course poster template is built at the Expo's
standard size, 48 by 36 inches, and its hidden second page is a guide to the print
spec and the panels. It comes with M10: your M10 submission includes a poster draft
built on it, which M11 revises and M13 locks. The course poster rubric grades the
locked poster at M13. Both are posted on the course platform with the poster
milestones, and they also live in the course repository:

- [Course poster template (.pptx)](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/project/poster/honr464_poster_template.pptx)
- [Course poster rubric (PDF)](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/project/poster/Evidence_Driven_Research_Poster_Rubric.pdf)

**What the Expo asks.** The Fall Expo's poster guidelines set the physical frame: no
wider and no taller than 4 feet, with a standard board **4 feet wide by 3 feet
tall**, set by the campus wide-format printers. You bring the printed poster to
check-in, and magnets are provided to mount it.

**How the Expo judges.** The Office of Undergraduate Research publishes the criteria
its judges use. Each one maps onto something this course already asks of you:

| The judges look for... | ...which on your board means |
|---|---|
| The purpose of the project, and why it matters to a broader audience | The title, the research problem, and the question, in plain words |
| Next steps based on progress to date | The conclusion and the next study, named specifically |
| Organized content; visual aids designed for understanding, labeled, without errors | The six principles, the read path, and the traceability sweep |
| Visual aids that support the key points | One headline figure carrying the claim, with its uncertainty |
| The method or approach used | Design and data, with the pathway named in plain words |
| Literature referenced; the work positioned within previous studies | What is known and what you add, from your verified evidence map |
| Inviting and answering questions; time used to emphasize key concepts | Your three pitches and the defend-or-concede call |
| Acknowledgments, at least the mentor | The acknowledgments block |

**Resources:**

- [Fall Expo poster guidelines](https://www.purdue.edu/undergrad-research/conferences/fall/guidelines.php):
  size, printing, mounting, and the AI guidance.
- [Judge evaluation criteria](https://www.purdue.edu/undergrad-research/conferences/Judge_EvaluationCriteria.pdf),
  from the Office of Undergraduate Research.
- [Professor Moreira's mentored projects](https://davi-moreira.github.io/applied_projects.html),
  with posters from past Purdue research conferences.

> *The judges' criteria and the course rubric overlap heavily, but they are
> separate documents. Read both.*

---

## 12. Planning Your Poster: Visual Hierarchy and Layout

A poster has hierarchy at **three layers**: the layout (what sits where), the
typography (what is loudest), and the content (what matters most). All three have to
agree, and they fail independently. Great content in a single column buries the
headline below the fold. Three tidy columns in five fonts make the eye recalibrate
at every block. A clean layout with unfocused content gives the reader nothing to
remember.

**Storyboard before software.** Draw your blocks on paper, in the order a visitor's
eye travels them, before you open the template. A poster laid out inside the
software takes the shape the software offers. A poster laid out on paper takes the
shape the argument needs.

The cell below draws a schematic of one sound layout at the Expo's standard size,
with the six required blocks outlined and the **read path**, the route a
visitor's eye follows, drawn across it. Expect the headline result to sit in the
center column, where the eye rests longest.

In [ ]:
# A schematic research poster at the Expo's standard size (4 ft wide by 3 ft tall).
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 9.75))
ax.set_xlim(0, 48)
ax.set_ylim(0, 36)
ax.axis("off")
ax.add_patch(plt.Rectangle((0, 0), 48, 36, fill=False, ec=DARK, lw=1.5))

def block(x, y, w, h, text, face="#F3F3F3", edge="#AAAAAA", size=10, weight="normal"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.2", fc=face, ec=edge))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=size, fontweight=weight)

REQ = dict(edge=DARK)                     # a dark edge marks the six required blocks
block(1, 31, 46, 4, "TITLE: the research problem and the answer\nauthor · mentor · course · Purdue",
      face="#E4E4E4", size=13, weight="bold", **REQ)
block(1, 21.2, 14.6, 8.8, "The question\nand why it matters\n(kind and reach on the compass)", **REQ)
block(1, 11.4, 14.6, 8.8, "What is known\nand what you add\n(your verified evidence map)")
block(1, 1, 14.6, 9.4, "Design and data\npathway · sample · measurement\nanalysis · how uncertainty\nwas computed", **REQ)
block(16.7, 9.2, 14.6, 20.8, "THE RESULT\n\nheadline figure\nwith its uncertainty\nin the same glance",
      face="#D9E8F5", edge=ACCENT, size=13, weight="bold")
block(16.7, 1, 14.6, 7.2, "Does it survive?\nthe stress test from Studio 8")
block(32.4, 21.2, 14.6, 8.8, "The boundary\nwhat the answer does NOT show\nlimitations · uncertainty", **REQ)
block(32.4, 11.4, 14.6, 8.8, "Conclusion\nand the next study")
block(32.4, 1, 14.6, 9.4, "AI-use disclosure\nethics and permissions\nacknowledgments · references\nQR code", **REQ)

# The read path, drawn through the blocks' top-left corners so it never covers text:
# title, question, result, boundary, conclusion, disclosure.
path = [(2.2, 34.1), (2.2, 29.1), (17.9, 29.1), (33.6, 29.1), (33.6, 19.3), (33.6, 9.5)]
xs, ys = zip(*path)
ax.plot(xs, ys, color=ACCENT, lw=2, ls="--", alpha=0.75, zorder=0)
for i, (x, y) in enumerate(path, start=1):
    ax.scatter(x, y, s=260, color=ACCENT, zorder=3)
    ax.text(x, y, str(i), color="white", ha="center", va="center", fontsize=9, fontweight="bold", zorder=4)
ax.set_title("A sound evidence-driven research poster: dark-edged blocks are required "
             "(the result is the blue one); numbered circles trace the read path", loc="left", fontsize=11)
plt.tight_layout()
plt.show()


**Reading the output.** The eye enters at the title, takes the question on the
left, and rests on the headline result in the center, which is why that block carries
the accent and the most space. It then moves right to the boundary before it reaches
the conclusion, so no visitor meets your conclusion without first passing its limits.
The disclosure block closes the path, where a judge who wants to check your process
will look for it.

**Layout and design guidelines.** Treat these as defaults you adjust on purpose, and
know which layer you are touching when you break one.

- **Keep it simple.** Rely on figures, short paragraphs, and a few bullets. One font
  family, consistent alignment, one accent color.
- **White space is your friend.** Group related blocks, leave margins, and aim for
  roughly 60% graphics and 40% text.
- **Text sizes at print scale.** The course template sets the minimums: title at
  least 72 pt, headings at least 32 pt, body text at least 24 pt, captions and
  references at least 18 pt. Go larger wherever the space allows.
- **The four-foot test.** Print one block at full size, tape it to a wall, and step
  back four feet. If you cannot read the smallest label, neither can the judge.
- **The thumbnail test.** View the whole poster at 25% scale. If the headline, the
  design, and the boundary are not findable in ten seconds, one of the three layers
  is failing, most often typography first and content second.

---

## 13. Planning Your Poster: Organizing Your Content

Research posters across fields share one architecture, because it follows the order
a trained reader evaluates evidence: why should I care, what did you find, why
should I believe it, and where does it stop. Here it is, adapted to an
evidence-driven project. Rows marked **(required)** are the six blocks every draft
must contain, even while a block is still rough.

<table>
<tr><th>#</th><th>Section</th><th>What goes here</th></tr>
<tr><td>1</td><td><strong>Title and authors</strong> (required)</td><td>A title that names the research problem and the answer, readable across the room. Your name, your mentor, the course, Purdue.</td></tr>
<tr><td>2</td><td><strong>Abstract or introduction</strong></td><td>The structured paragraph you entered for the Expo, updated with your verified result (Exercise 2).</td></tr>
<tr><td>3</td><td><strong>The research problem</strong></td><td>Why the question matters, and to whom, in plain words.</td></tr>
<tr><td>4</td><td><strong>The question and its compass position</strong> (required)</td><td>One sentence, with its kind (descriptive or causal) and its reach (the data at hand, a population, or unseen cases).</td></tr>
<tr><td>5</td><td><strong>What is known and what you add</strong></td><td>The literature's main conclusion, from sources you read and verified, and the value your answer adds to it.</td></tr>
<tr><td>6</td><td><strong>Design and data</strong> (required)</td><td>The pathway in plain words, the sample and how it was drawn or assigned, the measures, the analysis, and how the uncertainty was computed (including any clustering).</td></tr>
<tr><td>7</td><td><strong>The result, with its uncertainty</strong> (required)</td><td>The headline figure and estimate, the interval in the same glance, and the licensed comparison beside it.</td></tr>
<tr><td>8</td><td><strong>Does it survive?</strong></td><td>The stress test from Studio 8: the alternative specification, check, or rival explanation, and what happened to the answer.</td></tr>
<tr><td>9</td><td><strong>The boundary</strong> (required)</td><td>What the answer does not show, the crossing your design did not pay for, and the main limitations.</td></tr>
<tr><td>10</td><td><strong>Conclusion and next study</strong></td><td>The bounded conclusion in one sentence, and the specific study that would settle what remains open.</td></tr>
<tr><td>11</td><td><strong>AI-use disclosure, ethics, acknowledgments, references</strong> (required)</td><td>Tool, purpose, how you changed its output, and how you verified it; the ethics or permission status; your mentor and anyone who helped; verified references and the data attribution; a QR code.</td></tr>
</table>

> **A question that often comes up here:** *"Can I drop a section if my project does
> not need it?"* Yes, if you can defend the omission to a judge in one sentence. The
> most common defensible cut is folding the research problem into the question when
> the two say nearly the same thing. The six required blocks are never optional, and the
> most common indefensible cut is the disclosure and references block "for space,"
> which signals a layout problem, not a content problem.

### Your Twelve-Week Chain, on One Page

A poster is not a new piece of work. It is the chain you built studio by studio,
compressed into blocks a stranger can read. The cell below draws that mapping:
which studios produced which part of the board. Expect every block to trace back to
work you have already done, and the AI Research Ledger to run underneath all of
them.

In [ ]:
# Which studios produced which poster blocks: the poster compresses the chain.
chain = [("Title and question", "Studios 1-2\nframe the inquiry,\ndeclare the question"),
         ("What is known", "Studio 3\nverified evidence\nmap"),
         ("Design and data", "Studios 4-6\ndeclare, diagnose,\ngovern the data"),
         ("The result and\nwhether it survives", "Studios 7-8\nreproducible analysis,\nstress test"),
         ("The boundary and\nthe conclusion", "Studio 9\nwrite, bound,\ndisclose")]
fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 50)
ax.set_ylim(0, 14)
ax.axis("off")
for i, (blk, studios) in enumerate(chain):
    x = 1 + i * 10
    ax.add_patch(plt.Rectangle((x, 7.6), 8.2, 5, fc="#D9E8F5", ec=ACCENT))
    ax.text(x + 4.1, 10.1, blk, ha="center", va="center", fontsize=10, fontweight="bold")
    ax.add_patch(plt.Rectangle((x, 2.6), 8.2, 4.2, fc="#F3F3F3", ec="#AAAAAA"))
    ax.text(x + 4.1, 4.7, studios, ha="center", va="center", fontsize=8.5)
    if i < len(chain) - 1:
        ax.annotate("", xy=(x + 9.9, 10.1), xytext=(x + 8.3, 10.1),
                    arrowprops=dict(arrowstyle="->", color=DARK))
ax.add_patch(plt.Rectangle((1, 0.4), 48.2, 1.6, fc="#EDEDED", ec="#AAAAAA"))
ax.text(25, 1.2, "Every studio: the AI Research Ledger  →  the poster's AI-use disclosure block",
        ha="center", va="center", fontsize=9.5)
ax.text(1, 13.4, "Poster blocks (top) and the studios that built them (below). "
        "Studio 10 adapts the whole chain to the venue.", fontsize=10)
plt.show()


> **A question that often comes up here:** *"Do I have to draw this chain on my
> poster?"* No. But a reader should be able to reconstruct it from your blocks: the
> question you declared, the design you diagnosed, the data you governed, the analysis
> anyone can rerun, and the boundary you wrote. If a block on your board has no studio
> behind it, ask where its content came from before a judge does.

---

## 14. Designing for Evidence-Driven Research: The Special Case

Most poster advice is the same in every field. This section is about what changes
because your poster carries a **research claim**: a statement whose verb, reach, and
uncertainty are fixed by a design. These are the details a judge who knows research
looks for first. They fall into two groups: how you **report** the answer, and how
you **show that it is honest**.

**Reporting the answer: three things judges look for.**

First, **name your design in plain words.** "A randomized field experiment," "a
descriptive comparison in survey data," "an observational comparison with matched
groups." Do not make the reader reverse-engineer the design from the numbers.

Second, **report the estimate with its interval, and say what the interval
accounts for.** Name what you are estimating, the warrant that lets the sample speak
for anything beyond itself, and the dependence your interval respects. "A 2.8-point
gap, 95% interval −1.0 to +6.7, clustered by street" reads as careful work. "Turnout
rose 2.8 points" reads as overclaiming.

Third, **show the licensed comparison**: the control group, the benchmark, the
published estimate from your evidence map, or the baseline. A number without a
comparison invites the reader to supply one.

**Showing it is honest: three more things.**

Fourth, **the headline verb matches the compass position.** A descriptive or
observational design reports "is associated with" or "reports more," never "causes"
or "builds." A population claim needs a sampling warrant; a claim about unseen cases
needs a held-out check.

Fifth, **every number traces to the cell that produced it.** Your claim-evidence
table is the map, and the traceability sweep checks it number by number.

Sixth, **the AI-use disclosure is on the board.** The tool, its purpose, how you
changed its output, and how you verified it, drawn from your AI Research Ledger. The
Expo requires it, and EDR|AI treats it as part of the evidence.

**The two mistakes that hurt research posters most.** The first is a number with
no uncertainty, which reads as overclaiming. The second is a verb bigger than the
design, which reads as not understanding your own study. Both are quick fixes: your
intervals came out of Studios 7 and 8, your boundary sentence came out of Studio 9,
and both rubrics reward you for showing them.

> **A question that often comes up here:** *"Why do judges care so much about the
> interval and the verb?"* Because together they answer the only two questions a
> skeptical reader really has: "how sure are you?" and "how far does this go?" The
> interval answers the first by showing how much the number could wobble. The verb
> answers the second by saying whether you found a pattern, a population fact, or a
> cause. A strong number with no interval and an oversized verb looks like a claim
> with no evidence behind it, however good the number is.

---

## 15. Crafting a Clear Narrative

The poster is the medium. The claim is the message. Two reminders before you write:
one about tone, one about discipline.

**Tone: write the way a careful person talks.** Short sentences, one idea each.
Active voice ("I compared," not "a comparison was performed"). Concrete nouns over
abstractions. Every technical term defined once, in plain words, the first time it
appears. Honest uncertainty, stated once and clearly, instead of a hedge in every
sentence.

**Discipline: every word competes for ninety seconds.** A poster holds about 600
words. Each adjective you keep costs a reader's attention, and each sentence that
repeats another costs you the visitor who was about to read your boundary.

**Your abstract is already written, so improve it.** You entered a title and a
structured abstract for the Expo, and the conference lets you update it until its
final-updates deadline. The same structure works as your poster's introduction. It
is **one paragraph** in four parts, with the title limited to 125 characters and the
abstract to 2,000 characters, spaces included:

1. **Question and answer:** the research question; the answer, no stronger than your
   route licenses; the background, with the literature's main conclusion and your
   main hypothesis; and your contribution, meaning how your answer adds value to that
   conclusion.
2. **The data,** with what your exploration showed, labeled as exploratory.
3. **The design:** study design, sample size, data-collection tools, and analysis.
4. **The finding:** the primary result with its numbers and uncertainty, then the
   broader impact, the limitations, and the next step.

The title names the research problem and the answer. And the abstract is yours: the
Expo's AI guidance rejects AI-generated abstracts, so a tool may review your
paragraph, never write it.

> **A question that often comes up here:** *"What if my result is not final yet?"*
> Write it as an expectation, labeled as one ("I expect," never "I find"), and say
> what it rests on. When the verified result exists, it replaces the expectation in
> the same submission. Updating a number takes a minute; restructuring a paragraph
> you never wrote takes an evening you will not have.

---

## 16. Effective Figures and Tables on a Poster

Everything in Sections 3 to 8 applies here, with one catch: a poster is read from
four feet away, once, with no second chance. Your reader cannot squint at a tiny
label, hover for a tooltip, or scroll back to a caption.

For **figures**, four rules do most of the work.

First, **label every axis, with units.** An unlabeled axis is a question the reader
must answer before reading anything else.

Second, **one accent, backed by a second channel.** Spend the accent on the data that
carries your claim, and back it with a label or a pattern so the message survives
grayscale and color-blind readers.

Third, **put the uncertainty on the mark.** The interval belongs on the figure, in the
same glance as the estimate, not in a footnote.

Fourth, **write captions that state the bounded conclusion, not the topic.**
"Respondents who rate the government responsive report about half a point more police
trust; the intervals overlap" tells the reader what the evidence says and where it
stops. "Police trust by responsiveness" repeats the axis labels.

For **tables**, one test: could a figure make the same point faster? If so, use the
figure. A table earns poster space only when the reader must look up an exact value,
such as a small table of sample sizes.

The cell below builds a poster-ready headline figure from nb10's comparison, sized
and lettered for print, with the caption written as a bounded conclusion. It
reproduces nb10's seeded subsample exactly. Expect the same numbers you met there:
4.31 ± 0.33 against 4.83 ± 0.32.

In [ ]:
# A poster-ready headline figure: nb10's comparison, sized and lettered for print.
w10 = np.random.default_rng(SEED)                      # the same draws nb10 made
low = lapop.loc[lapop["govt_responsive"] <= 3, "trust_police"].to_numpy()
high = lapop.loc[lapop["govt_responsive"] >= 4, "trust_police"].to_numpy()
a = low[w10.choice(low.size, 150, replace=False)]
b = high[w10.choice(high.size, 150, replace=False)]
m_a, m_b = a.mean(), b.mean()
h_a, h_b = (1.96 * x.std(ddof=1) / np.sqrt(len(x)) for x in (a, b))
print(f"Rated 1-3: {m_a:.2f} ± {h_a:.2f}   Rated 4-7: {m_b:.2f} ± {h_b:.2f}   gap {m_b - m_a:.2f}")
assert round(m_a, 2) == 4.31 and round(m_b, 2) == 4.83, "should match nb10 exactly"

with plt.rc_context({"font.size": 16}):
    fig, ax = plt.subplots(figsize=(9, 6.5))
    bars = ax.bar(["Rated government\n1 to 3", "Rated government\n4 to 7"], [m_a, m_b],
                  yerr=[h_a, h_b], capsize=10, color=[GREY, ACCENT], hatch=["", "//"],
                  edgecolor="white", width=0.55)
    for bar, m, h in zip(bars, [m_a, m_b], [h_a, h_b]):
        ax.text(bar.get_x() + bar.get_width() / 2, m + h + 0.15, f"{m:.2f}", ha="center",
                fontweight="bold")
    ax.set_ylim(1, 7)
    ax.set_ylabel("Mean trust in the police\n(1 = none, 7 = a lot)")
    declutter(ax, keep_left=True)
    fig.text(0.02, 0.01, "Respondents who rate the government responsive report about half a point "
             "more trust in the police;\nthe 95% intervals overlap. Teaching resample, 150 per group: "
             "an association, not a claim about Brazil.", fontsize=11, color=DARK)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    plt.show()


**Reading the output.** The figure carries every rule at once: the full 1-to-7
scale, the values printed on the marks, the intervals drawn on top, the accent backed
by a hatch pattern, and a caption that says what the evidence shows and where it
stops. Its lettering is sized for print: at 4 feet wide, a figure like this one
should take roughly a third of a column, with its labels at least 24 pt and its
caption at least 18 pt.

**How many figures?** Three to five. One **headline figure** for the main result, one
**design figure** (how the sample was drawn or assigned, or how a key concept was
measured), and one **stress-test figure** (the result under an alternative
specification, or the rival explanation checked). Past five, figures start competing
for attention, which is the hierarchy problem of Section 6 at the scale of a whole
board.

**Extend the poster beyond the wall with a QR code.** A **QR code** is the small
square barcode a phone camera turns into a web link. Print one in a corner and you
give curious readers a door to everything that did not fit: your reproducible
package, the full regression table, your claim-evidence table, or your AI disclosure
in full. Put a one-line label under it ("Scan for the full analysis" beats a bare
square), keep the link short and stable, and test the printed code with two or three
phones. The poster still has to tell the whole story without it, because most
readers will never scan.

> **A question that often comes up here:** *"What should my QR code point to?"* The
> destination that rewards a careful reader most. For a research poster that is
> usually your reproducible package or a short page with the full analysis, the
> claim-evidence table, and the AI disclosure. Never send a phone to a raw,
> unlabeled file, and never to anything that holds identifiable information about
> the people in your data.

---

## 17. Results and Interpretation

This is the block judges study longest, so make it easy on them. Show three things,
in this order.

First, the **headline estimate** with its interval: the single number that answers
your question.

Second, the **licensed comparison** that makes the number evidence: the control, the
benchmark, or the published estimate, on the same figure.

Third, the **stress test**: what happened to the answer when you changed the
specification, checked a rival explanation, or ran the check your Studio 8 audit
required. If the answer moved, say how far.

That is the whole results block: one headline figure, one short paragraph, and a
small stress-test figure or a single sentence. The temptation is to show every
estimate from every notebook. Resist it. The extra estimates belong behind the QR
code.

**Showing your thinking.** Judges reward this more than most presenters expect. They
want to see that you understand *why* the answer came out as it did and where it is
weak: which subgroup drives it, which measure is noisiest, which rival explanation you
could not rule out. Naming your study's weak spot is not a confession. It is a
credibility builder, and it is exactly what your boundary block is for.

> **A question that often comes up here:** *"Should I put the regression table or the
> coefficient plot on the poster?"* The coefficient plot, with its intervals and a
> reference line at zero, as Section 4.1 showed. Put the full table behind the QR
> code for the reader who wants exact values.

---

## 18. Conclusion and the Next Study

The two short paragraphs at the end of your read path carry more weight than their
size suggests. The **conclusion** is the line a visitor repeats to a friend the next
day, so it must be concrete and bounded. The **next study** shows that you can see
past this project to the question it leaves open. Both are short, about 50 words
each, and both deserve more editing than anything else on the board.

**A strong conclusion does three things in three sentences.** It restates the finding
in plain words a non-specialist can repeat ("respondents who rate the government
responsive report about half a point more trust in the police"). It states the
boundary ("an association in a teaching sample, not an effect"). And it says how sure
you are ("the two 95% intervals overlap, so the gap is small and uncertain"). Compare
it with "the results were significant," which is forgettable and tells a reader
nothing they can use.

**A strong next study is specific.** "Rerun the comparison on the original survey
file with its design weights and municipality clusters, then test whether the gap
holds within each region" is a study someone could start tomorrow. "Future research
should explore this further" signals you ran out of ideas. The best next studies come
straight from your boundary: the crossing your design did not pay for is the study
that would pay for it.

> **A question that often comes up here:** *"How do I write the conclusion if my
> numbers might still change?"* Draft it now. The shape (finding, boundary,
> confidence) does not change if an estimate moves by a decimal once your
> reproduction run is final. Swapping in the final number takes a minute.

---

## 19. Final Touches

Acknowledgments, references, and proofreading are not afterthoughts; they are points
on both rubrics, and the Expo's judges check acknowledgments directly.

**Acknowledgments and references are scored.** Thank your mentor first (the judges'
criteria require at least the mentor), then anyone else who helped and any funding
source. Credit your data with its attribution line. List only references you read and
verified, in one consistent format: a citation your AI suggested is a lead to check
in the Verification Guide, never a reference to print.

**The traceability sweep is where the last points hide.** Check every number on the
board against the cell that produced it, as the poster milestones ask. A poster that says 0.52 when
your notebook printed 0.51 quietly undermines the whole board, and copying from your
own rounded prose is the usual cause. Make sure every image is sharp at print size,
because a blurry figure enlarged to 4 feet looks worse than no figure at all.

**Rehearse it on a non-specialist.** Print a draft at full size, tape it to a wall,
stand four feet back, and ask someone outside the course to find your headline and
your boundary. If they cannot find both in thirty seconds, the layout has buried
them. Do this at least two days before the lock, so you have time to redesign rather
than retype.

---

## 20. Presenting Your Poster

At the Expo the board is half the work and you are the other half. The board must
stand on its own when nobody is there; you are the guided tour with a human voice.
EDR|AI ch. 34 and ch. 35 teach the spoken half in depth, and nb10's second lecture
rehearses it. The habits below are the ones that matter most on the floor.

- **Prepare three pitches**, 30 seconds, 90 seconds, and 2 minutes, each built
  **boundary-first**: write the boundary sentence before anything else, because it is
  the beat that pressure deletes. The 30-second version is one bounded sentence. The
  90-second version adds the design and the key limitation. The 2-minute version
  walks the read path.
- **Invite questions**, and open with one of your own ("Are you familiar with...?").
  The judges look for both inviting and answering questions.
- **Defend, concede, or say you do not know.** Defend when a question asks for
  something your evidence already licenses. Concede, narrowing the claim on the spot,
  when it asks for a crossing your design never paid for. When you cannot answer, give
  an honest don't-know: what you cannot say, what you can stand on, and the study that
  would settle it.
- **Stand beside the board**, not in front of it, and use your hand to bring the
  listener's eye to the block you are talking about.
- **Arrive early** with your printed poster; magnets are provided for mounting.
- **Dress:** the Expo encourages business casual without requiring it, and the
  Purdue Career Closet at the Center for Career Opportunities can help if you need
  clothing.

---

## 📝 Exercise 2: Storyboard, Outline, Title, and Abstract

**Task:** turn your bounded note into the plan for your board. This is the
highest-leverage page you can write this week: the outline locks the layout of the
poster draft you build on the template for M10, and the abstract is the paragraph
the most people will read.

1. **Storyboard on paper.** Sketch the six required blocks in the order a visitor's eye
   travels them, and mark where the headline result sits. Describe it in one line in
   the answer cell, or paste a photo link.
2. **Outline.** Fill the section table in the answer cell, one or two sentences per
   row, in your own words. Where a row has no answer yet, write *open* and name the
   milestone that will close it.
3. **Title.** At most 125 characters, spaces included, naming the research problem
   and the answer.
4. **Abstract.** Update the abstract you entered for the Expo so it matches your
   verified result: one paragraph, at most 2,000 characters, in the four parts of
   Section 15. A result that is not final is written as an expectation.
5. **Boundary sentence.** One sentence, on its own line, saying what your answer does
   not show.
6. **Read it aloud.** Split any sentence longer than thirty words.

### 🧑‍⚖️ Human-Only Checkpoint

*(AI closed. These words must be yours: the Expo's guidance rejects AI-generated
abstracts, and EDR|AI never delegates a headline.)*

Write steps 3, 4, and 5 with your AI closed. Only after all three exist on the page do
you open a tool, and then only to review what you wrote.

> 💡 **AI Prompt:** "Here are my poster title, my abstract, and my boundary
> sentence, word for word: [paste all three]. My compass position is [descriptive or
> causal], reaching [the data at hand, a population, or unseen cases], and my
> headline estimate is [the number with its interval]. Act as a skeptical conference
> judge. (1) List every word in the title or abstract that claims more than that
> position and interval license. (2) Check whether the abstract follows this order:
> question, answer, background with the literature's main conclusion and my
> hypothesis, contribution; data; design and analysis; finding with its uncertainty,
> impact, limitations, and next step. Name any part that is missing. Rewrite
> nothing."
>
> **After running, verify (counters *sycophantic agreement*: praise for a paragraph
> that reviews your confidence, not your evidence):**
> - [ ] Check each flagged word against your claim-evidence table. Keep a word the
>   evidence licenses, and write down why.
> - [ ] If the tool flags nothing, push back once and ask for the single word it
>   would cut first. A review with no findings is a compliment, not a review.
> - [ ] Count the characters yourself with the cell below; tools miscount.
> - [ ] Any wording you adopt from the tool is editing support: say so in your AI-use
>   disclosure.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

Run the cell below after pasting your own title and abstract between the quotes.
Expect two character counts and a count of line breaks, which should be zero.

In [ ]:
# Paste your title and abstract between the quotes, then run. The Expo counts spaces.
my_title = """Paste your title here"""
my_abstract = """Paste your abstract here, as one paragraph"""

print(f"Title    : {len(my_title.strip()):>5,} characters (limit 125)")
print(f"Abstract : {len(my_abstract.strip()):>5,} characters (limit 2,000)")
print(f"Line breaks inside the abstract: {my_abstract.strip().count(chr(10))} (one paragraph means zero)")


### YOUR ANSWER HERE:

**Storyboard (one line, or a photo link):**

**Section outline:**

| # | Section | Draft (one or two sentences) |
|---|---|---|
| 1 | Title and authors | |
| 2 | Abstract or introduction | *(see below)* |
| 3 | The research problem | |
| 4 | The question and its compass position | |
| 5 | What is known and what you add | |
| 6 | Design and data | |
| 7 | The result, with its uncertainty | |
| 8 | Does it survive? | |
| 9 | The boundary | |
| 10 | Conclusion and next study | |
| 11 | AI-use disclosure, ethics, acknowledgments, references | |

**Title (at most 125 characters):**

**Abstract (one paragraph, at most 2,000 characters):**

**Boundary sentence:**

**What the AI review flagged, and what I kept or changed, with the reason:**

---

## 21. Additional Material

None of these is required. Each repays an hour when you are sharpening a figure.

- [The Python Graph Gallery](https://python-graph-gallery.com/) and
  [The R Graph Gallery](https://r-graph-gallery.com/), chart ideas organized by type,
  each with its code.
- [The Matplotlib examples gallery](https://matplotlib.org/stable/gallery/index.html),
  for the exact call behind a layout you want to copy.
- Erren and Bourne's (2007) ten rules for a good poster presentation, short and
  practical; the full reference is in Section 23.
- Crameri, Shephard, and Heron (2020) on the misuse of color in science
  communication, the reason this notebook spends one accent and backs it with a
  second channel.

---

## 22. Wrap-Up: Key Takeaways

**The six principles you can audit any figure against:**

1. **Context matters:** every number needs the comparison its design licenses.
2. **Visualization derives from data:** the data's shape and scale pick the chart.
3. **Less is more:** delete ink that encodes nothing, and never the uncertainty.
4. **Hierarchy:** the eye is not democratic; choose what wins it.
5. **Beauty counts:** a calm palette keeps the reader long enough to hear the claim.
6. **Telling your story:** the last line is your bounded headline.

**The rule above them:** the picture may make the claim easier to see. It may never
make the claim bigger.

**The poster's six required blocks, in read-path order:** title and authors →
question → design and data → result with uncertainty → boundary → AI-use disclosure.

> *"Cover your headline figure's title. If a stranger cannot restate your bounded
> sentence from what is left, the figure is not ready."*

**Where this goes next.** Friday's studio builds your M10 artifact, including your
first poster draft on the course template; bring the figure you audited in Exercise 1
and the outline and abstract from Exercise 2. Next week's poster production revises
that draft for M11, where it goes through structured peer review, and then into the
terminal poster lock at M13. Two rebuilds of your
headline figure and three drafts of your abstract are a realistic plan. Stop when a
non-specialist reads the board top to bottom in ninety seconds and restates the
bounded result in one sentence. That is the test the Expo floor will run.

### 🛡️ Defend Your Decision

Close the notebook out loud, one line each:

1. **The claim I will put on my board:** the bounded headline, as a visitor will read it.
2. **Its boundary:** the compass position it speaks from, and the crossing it does not license.
3. **My uncertainty and limitations:** how much the number could wobble, and the main limit.
4. **AI check:** what I delegated in this notebook, and the specific check that decided what I kept.

### YOUR ANSWER HERE:

**1. The claim I will put on my board:**

**2. Its boundary:**

**3. My uncertainty and limitations:**

**4. AI check:**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in your AI Research Ledger. One worked row models
the week's discipline: **you write the words first, the tool attacks them, and your
own evidence settles the argument.** Add a row for each prompt you actually ran, and
if you delegated nothing, log that decision too.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Find every word in my title and abstract that claims more than my interval licenses | your AI | "Here are my poster title, my abstract, and my boundary sentence, word for word... Act as a skeptical conference judge... Rewrite nothing." | Flagged three words; two overreached, one was a term my design does license | Cut the two overreaching words; kept the licensed one and noted why | Read each flag against my claim-evidence table; recounted characters with the counting cell | The check cannot tell me whether my boundary is the right one for my design | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

## 23. Sources & Provenance

**Provenance of this notebook:**

- *Professor Moreira's QM 47400 Predictive Analytics notebook 17, "Elements of Data
  Communication and Poster Design" | the six principles, the cleanup sequence, the
  poster architecture, and the two exercises | adapted: rewritten for evidence-driven
  research, with every figure redrawn in code from course data or a seeded simulation
  (seed 464)*
- *EDR|AI ch. 32 'Research Posters' and ch. 33 'Poster Criticism' | figure honesty,
  the claim boundary, uncertainty on the page, redundant encoding, the gallery-walk
  audits | applied*
- *EDR|AI ch. 34 'Research Pitches, Talks, and Seminars' and ch. 35 'Difficult
  Questions and Uncertainty' | the boundary-first pitch, defend or concede, the honest
  don't-know | referenced*
- *Course milestones M10, M11, and M13 | the six required poster blocks, the
  storyboard, the traceability sweep, the reach check | referenced*
- *Purdue Office of Undergraduate Research, Fall Expo poster guidelines and judge
  evaluation criteria | size, printing, mounting, AI guidance, judging criteria |
  summarized from the official pages*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock & Humphreys,
MIT License), companion to *Research Design in the Social Sciences* (2023).
`lapop_brazil` is a teaching resample of AmericasBarometer (LAPOP) Brazil interviews;
`foos_etal` is the replication file of the UK get-out-the-vote field experiment by Foos
and colleagues.

**Readings:**

- Moreira, D. *Evidence-Driven Research in the Age of AI* (EDR|AI), ch. 32
  [Research Posters](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part5-communicating/27-research-posters.html)
  and ch. 33
  [Poster Criticism](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part5-communicating/28-poster-criticism.html).
- Tufte, E. R. (2001). *The Visual Display of Quantitative Information* (2nd ed.).
  Graphics Press. The data-ink ratio.
- Knaflic, C. N. (2015). *Storytelling with Data*. Wiley. Decluttering, preattentive
  attributes, and emphasis, for presentations.
- Kastellec, J. P., & Leoni, E. L. (2007). Using graphs instead of tables in political
  science. *Perspectives on Politics*, 5(4), 755–771.
  [https://doi.org/10.1017/S1537592707072209](https://doi.org/10.1017/S1537592707072209)
- Erren, T. C., & Bourne, P. E. (2007). Ten simple rules for a good poster
  presentation. *PLOS Computational Biology*, 3(5), e102.
  [https://doi.org/10.1371/journal.pcbi.0030102](https://doi.org/10.1371/journal.pcbi.0030102)
- Crameri, F., Shephard, G. E., & Heron, P. J. (2020). The misuse of colour in science
  communication. *Nature Communications*, 11, 5444.
  [https://doi.org/10.1038/s41467-020-19160-7](https://doi.org/10.1038/s41467-020-19160-7)

<center>

Thank you!

</center>